Below is example code used to fine-tune the Detectron2 pre-trained segmentation base model. Please provide your own folder names and registered datasets.

### ⚠️ Google Colab Only Notebook

This notebook is designed to run in **Google Colab**. To start using this notebook, clone this GitHub repo in Colab:
```
!git clone https://github.com/Lynnicia/CryoEM_ultrastructures_top_model_decision_tree/tree/main.git
```
It requires:
- GPU runtime (CUDA) for speed tests
- Linux environment
- Colab-specific install steps (e.g., `!pip install`, `!git clone`, `from google.colab.patches import cv2_imshow`)

❌ Running locally (especially on Windows) may fail due to:
- Detectron2 installation issues
- CUDA/toolchain mismatches
- Missing compiled operators

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install roboflow --no-deps
!pip install pillow
!apt-get update
!apt-get install -y libavif-dev libaom-dev
!pip install --no-cache-dir pillow-avif-plugin pi-heif
!pip install filetype

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="[your_api_key]")
project = rf.workspace("[workspace_name]").project("[project_name]")
version = project.version([version_number])
dataset = version.download("coco-segmentation")

In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

In [ ]:
#copy jsons into the main folder for Detectron2
!cp /content/[roboflow_project_folder_name]/train/_annotations.coco.json /content/[roboflow_project_folder_name]/t.json

!cp /content/[roboflow_project_folder_name]/valid/_annotations.coco.json /content/[roboflow_project_folder_name]/v.json


In [ ]:
#inspect json format for TRAIN and delete bacteria category to only keep OM and IM categories

import json

with open("/content/[roboflow_project_folder_name]/t.json") as f:
    coco = json.load(f)

for c in coco["categories"]:
    print(c["id"], c["name"])

cats = sorted(coco["categories"], key=lambda x: x["id"])


# Update annotations

BAD_ID = 0

coco["categories"] = [
    c for c in coco["categories"] if c["id"] != BAD_ID
]

coco["annotations"] = [
    ann for ann in coco["annotations"]
    if ann["category_id"] != BAD_ID
]

with open("/content/[roboflow_project_folder_name]/train.json", "w") as f:
    json.dump(coco, f, indent=2)

for c in coco["categories"]:
    print(c["id"], c["name"])


In [ ]:
#inspect json format for VALID and delete bacteria category

import json

with open("/content/[roboflow_project_folder_name]/v.json") as f:
    coco = json.load(f)

for c in coco["categories"]:
    print(c["id"], c["name"])

cats = sorted(coco["categories"], key=lambda x: x["id"])


# Update annotations

BAD_ID = 0

coco["categories"] = [
    c for c in coco["categories"] if c["id"] != BAD_ID
]

coco["annotations"] = [
    ann for ann in coco["annotations"]
    if ann["category_id"] != BAD_ID
]

with open("/content/[roboflow_project_folder_name]/valid.json", "w") as f:
    json.dump(coco, f, indent=2)

for c in coco["categories"]:
    print(c["id"], c["name"])


In [ ]:
from detectron2.engine import HookBase
from detectron2.evaluation import inference_on_dataset, COCOEvaluator
from detectron2.data import build_detection_test_loader

class APHook(HookBase):
    def __init__(self, cfg, dataset_name, period=200):
        self.cfg = cfg
        self.dataset_name = dataset_name
        self.period = period
        self.best_ap = -1.0
        self.last_ap = None

    def after_step(self):
        it = self.trainer.iter
        if it % self.period != 0:
            return

        model = self.trainer.model
        model.eval()

        evaluator = COCOEvaluator(
            self.dataset_name,
            distributed=False,
            output_dir=self.cfg.OUTPUT_DIR,
        )

        val_loader = build_detection_test_loader(
            self.cfg,
            self.dataset_name
        )

        results = inference_on_dataset(model, val_loader, evaluator)

        ap = results["segm"]["AP"]
        self.last_ap = ap

        print(f"[VAL] iter={it} segm_AP={ap:.4f}")

        if ap > self.best_ap:
            self.best_ap = ap
            self.trainer.checkpointer.save("best_model")
            print(f"New best model saved (AP={ap:.4f})")

        model.train()

In [ ]:
from detectron2.engine import HookBase

class EarlyStoppingHook(HookBase):
    def __init__(self, ap_hook, patience=5, period=200, min_delta=0.001):
        self.ap_hook = ap_hook
        self.patience = patience
        self.period = period
        self.min_delta = min_delta

        self.best = -1.0
        self.counter = 0

    def after_step(self):
        it = self.trainer.iter

        # Only check at evaluation steps
        if it % self.period != 0:
            return

        val_ap = self.ap_hook.last_ap
        if val_ap is None:
            return

        print(f"[EARLYSTOP] iter={it} segm_AP={val_ap:.6f}")

        # Improvement
        if val_ap > (self.best + self.min_delta):
            self.best = val_ap
            self.counter = 0
            print("Improved. Counter reset.")
        else:
            self.counter += 1
            print(f"No improvement. Patience: {self.counter}/{self.patience}")

        if self.counter >= self.patience:
            print("Early stopping triggered ✅")
            raise StopIteration

In [ ]:
# function to globally register dataset in Detectron 2

from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

def reregister_coco(name, json_file, image_root):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

    register_coco_instances(name, {}, json_file, image_root)

In [ ]:
# ALWAYS rename dataset if data changes!!

for d in ["train", "valid"]:
    reregister_coco(
        f"[your_sample_name]_{d}",
        f"/content/[roboflow_project_folder_name]/{d}.json",
        f"/content/[roboflow_project_folder_name]/{d}"
    )

In [ ]:
td = DatasetCatalog.get("[your_sample_name]_train")[0]
print(td)
vd = DatasetCatalog.get("[your_sample_name]_valid")[0]
print(vd)

In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [ ]:
##################### Train the model ##########################

# Change to a known accessible directory
%cd /content/bacteria-thickness_additional-6

from detectron2.data import DatasetCatalog
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
import pandas as pd


cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")) # removed normalization, standard configuration used: https://deepwiki.com/facebookresearch/detectron2/7-model-zoo
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.MODEL.PIXEL_MEAN = [0.0, 0.0, 0.0] # disable normalization
cfg.MODEL.PIXEL_STD  = [1.0, 1.0, 1.0] # disable normalization
cfg.INPUT.MIN_SIZE_TRAIN = (1024,)
cfg.INPUT.MAX_SIZE_TRAIN = 1024
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 1024
cfg.DATASETS.TRAIN = ("[your_sample_name]_train",)
cfg.DATASETS.TEST = ("[your_sample_name]_valid",)
cfg.INPUT.MASK_FORMAT = "bitmask"
cfg.SOLVER.CHECKPOINT_PERIOD = cfg.SOLVER.MAX_ITER + 1  # load .pth
cfg.DATALOADER.NUM_WORKERS = 2


cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 2.5e-4
cfg.SOLVER.WARMUP_ITERS = 200
cfg.SOLVER.WARMUP_FACTOR = 0.4       # starts at 1e-4, ramps to 2.5e-4
cfg.SOLVER.STEPS = ()                 # no decay steps
cfg.SOLVER.GAMMA = 1.0               # no-op since no steps, but explicit is cleaner
cfg.MODEL.ROI_MASK_HEAD.RESOLUTION = 56

cfg.SOLVER.MAX_ITER = 8000
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)

ap_hook = APHook(cfg, "[your_sample_name]_valid", period=200)
early_stop = EarlyStoppingHook(ap_hook, patience=5, period=200, min_delta=0.001)

trainer.register_hooks([ap_hook, early_stop])

try:
    trainer.train()
except StopIteration:
    print("Training stopped early ✅")



In [ ]:
TARGET_FOLDER = "[target_folder_name]"
os.makedirs(TARGET_FOLDER, exist_ok=True)

# Validation Curves

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from detectron2.data import DatasetCatalog, build_detection_test_loader
from detectron2.data import detection_utils as utils
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model


# ===============================
# Build model and load weights
# ===============================

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

dataset_name = "[[your_sample_name]_valid]"

bacteria_metadata = MetadataCatalog.get(dataset_name)
bacteria_metadata.thing_classes = ["IM", "OM"]
dataset = DatasetCatalog.get(dataset_name)
dataset_dicts = DatasetCatalog.get(dataset_name)
cfg.MODEL.WEIGHTS = "[best_model.pth]"

# Match your dataset
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2  # change if needed
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 1024
# Inference settings
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.001
cfg.MODEL.DEVICE = "cuda"  # or "cpu"

model = build_model(cfg)
DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()



# ===============================
# COCO evaluation
# ===============================

evaluator = COCOEvaluator(
    dataset_name,
    tasks=("segm",),
    distributed=False,
    output_dir="./output"
)

val_loader = build_detection_test_loader(cfg, dataset_name)





In [ ]:
from detectron2.data import DatasetCatalog
import cv2
import numpy as np
from pycocotools import mask as mask_utils

model.eval()

classes = [0, 1]
all_true     = {0: [], 1: []}
all_probs    = {0: [], 1: []}


def decode_coco_segmentation(ann, height, width):
    seg = ann["segmentation"]

    if isinstance(seg, list):
        rles = mask_utils.frPyObjects(seg, height, width)
        rle = mask_utils.merge(rles)
        mask = mask_utils.decode(rle)

    elif isinstance(seg, dict):
        mask = mask_utils.decode(seg)

    else:
        raise ValueError(f"Unknown segmentation format: {type(seg)}")

    return (mask > 0).astype(np.uint8)

with torch.no_grad():

    for d in dataset_dicts:

        image = cv2.imread(d["file_name"])
        H, W = image.shape[:2]

        # Forward pass
        outputs = model([{
            "image": torch.as_tensor(image.transpose(2,0,1)).float().to(device),
            "height": H,
            "width": W
        }])[0]

        instances = outputs["instances"].to("cpu")

        # -----------------------
        # Build GT masks
        # -----------------------
        gt_im = np.zeros((H, W), dtype=np.uint8)
        gt_om = np.zeros((H, W), dtype=np.uint8)

        for ann in d["annotations"]:
            mask = decode_coco_segmentation(ann, H, W)

            if ann["category_id"] == 0:
                gt_im |= mask
            elif ann["category_id"] == 1:
                gt_om |= mask

        # -----------------------
        # Build probability maps
        # -----------------------
        prob_im = np.zeros((H, W), dtype=np.float32)
        prob_om = np.zeros((H, W), dtype=np.float32)

        for i in range(len(instances)):

            cls = instances.pred_classes[i].item()
            score = instances.scores[i].item()
            mask = instances.pred_masks[i].numpy()

            if cls == 0:
                prob_im[mask] = np.maximum(prob_im[mask], score)

            elif cls == 1:
                prob_om[mask] = np.maximum(prob_om[mask], score)

        # -----------------------
        # Store flattened pixels
        # -----------------------
        all_true[0].append(gt_im.ravel())
        all_true[1].append(gt_om.ravel())
        all_probs[0].append(prob_im.ravel())
        all_probs[1].append(prob_om.ravel())

for ch in [0,1]:
    all_true[ch] = np.concatenate(all_true[ch])
    all_probs[ch] = np.concatenate(all_probs[ch])

In [ ]:
import torch
import numpy as np
from detectron2.data import build_detection_test_loader, DatasetCatalog
from detectron2.data import detection_utils as utils

def build_yolo_style_pr_inputs(model, cfg, dataset_name, iou_thresh=0.5):

    model.eval()
    device = next(model.parameters()).device

    loader = build_detection_test_loader(cfg, dataset_name)
    dataset_dicts = DatasetCatalog.get(dataset_name)
    id_to_dict = {d["image_id"]: d for d in dataset_dicts}

    classes = [0, 1]
    all_true = {0: [], 1: []}
    all_probs = {0: [], 1: []}

    def mask_iou_matrix(pred_masks, gt_masks):
        if len(pred_masks) == 0 or len(gt_masks) == 0:
            return torch.zeros((len(pred_masks), len(gt_masks)), device=device)

        pred_flat = pred_masks.reshape(len(pred_masks), -1).float()
        gt_flat = gt_masks.reshape(len(gt_masks), -1).float()

        intersection = torch.matmul(pred_flat, gt_flat.T)
        area_pred = pred_flat.sum(dim=1).unsqueeze(1)
        area_gt = gt_flat.sum(dim=1).unsqueeze(0)
        union = area_pred + area_gt - intersection
        return intersection / (union + 1e-6)

    with torch.no_grad():
        for batch in loader:
            outputs = model(batch)

            for i, data in enumerate(batch):

                d = id_to_dict[data["image_id"]]

                # GT
                gt_instances = utils.annotations_to_instances(
                    d["annotations"],
                    (data["height"], data["width"])
                ).to(device)

                # Convert GT to bitmasks safely
                if len(gt_instances) > 0:
                    bitmasks = []
                    for poly in gt_instances.gt_masks.polygons:
                        mask = utils.polygons_to_bitmask(
                            poly,
                            data["height"],
                            data["width"]
                        )
                        bitmasks.append(torch.from_numpy(mask))
                    gt_masks_full = torch.stack(bitmasks).to(device)
                else:
                    gt_masks_full = torch.zeros(
                        (0, data["height"], data["width"]),
                        device=device
                    )

                pred_instances = outputs[i]["instances"].to(device)

                for ch in classes:

                    pred_c = pred_instances[pred_instances.pred_classes == ch]
                    gt_c = gt_instances[gt_instances.gt_classes == ch]

                    if len(pred_c) == 0:
                        continue

                    pred_masks = pred_c.pred_masks
                    scores = pred_c.scores

                    if len(gt_c) > 0:
                        gt_masks = gt_masks_full[gt_instances.gt_classes == ch]
                        ious = mask_iou_matrix(pred_masks, gt_masks)
                    else:
                        ious = torch.zeros((len(pred_masks), 0), device=device)

                    matched_gt = set()

                    for p_idx in range(len(pred_masks)):
                        if ious.shape[1] > 0:
                            max_iou, gt_idx = ious[p_idx].max(0)
                            if max_iou >= iou_thresh and gt_idx.item() not in matched_gt:
                                all_true[ch].append(1)
                                matched_gt.add(gt_idx.item())
                            else:
                                all_true[ch].append(0)
                        else:
                            all_true[ch].append(0)

                        all_probs[ch].append(scores[p_idx].item())

    # Convert to numpy arrays
    for ch in classes:
        all_true[ch] = np.array(all_true[ch])
        all_probs[ch] = np.array(all_probs[ch])

    return all_true, all_probs

    all_true, all_probs = build_yolo_style_pr_inputs(
    model, cfg, dataset_name
)

# ── Your existing plot code (unchanged) ───────────────────────────────────
labels = {0: "IM", 1: "OM"}
classes = [0, 1]

plt.figure(figsize=(7, 5))

for ch in classes:
    y_true = all_true[ch].ravel()
    y_pred = all_probs[ch].ravel()

    if y_true.size == 0:
        print(f"⚠️ Class {ch}: empty arrays, skipping PR curve")
        continue
    unique_vals = np.unique(y_true)
    if len(unique_vals) < 2:
        print(f"⚠️ Class {ch}: only one label present ({unique_vals.tolist()}), skipping PR curve")
        continue

    precision, recall, _ = precision_recall_curve(y_true, y_pred, pos_label=1)
    auprc = average_precision_score(all_true[ch], all_probs[ch])
    plt.plot(recall, precision, linewidth=1.5, label=f"{labels[ch]} {auprc:.3f}")

y_true_all = np.concatenate([all_true[ch].ravel() for ch in classes])
y_pred_all = np.concatenate([all_probs[ch].ravel() for ch in classes])
precision_all, recall_all, _ = precision_recall_curve(y_true_all, y_pred_all)
auprc_all = auc(recall_all, precision_all)
plt.plot(recall_all, precision_all, linestyle="-", linewidth=3, color="blue",
         label=f"all classes {auprc_all:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(False)
plt.ylim(0, 1)
plt.xlim(0, 1)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
save_path = os.path.join(TARGET_FOLDER, "VALID_Mask_PR_curve.png")
os.makedirs(TARGET_FOLDER, exist_ok=True)
plt.savefig(save_path, dpi=600, bbox_inches="tight")



# F1 curve

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(7.69,5))

labels = {0: "IM", 1: "OM"}

# ----- Plot per-class F1 curves -----
for ch in [0, 1]:
    precision, recall, thresholds = precision_recall_curve(
        all_true[ch], all_probs[ch]
    )
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    plt.plot(
        thresholds,
        f1[:-1],  # match threshold length
        label=labels[ch],
        linewidth=1.5
    )

# ----- Compute micro-average across all classes -----
all_true_flat = np.concatenate([all_true[0], all_true[1]])
all_probs_flat = np.concatenate([all_probs[0], all_probs[1]])

precision, recall, thresholds = precision_recall_curve(all_true_flat, all_probs_flat)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

max_idx = np.argmax(f1)
max_f1 = f1[max_idx]
best_conf = thresholds[max_idx]

plt.plot(
    thresholds,
    f1[:-1],
    label=f"all classes {max_f1:.3f} at {best_conf:.3f}",
    linestyle="-",
    color="blue",
    linewidth=3
)

# ----- Plot formatting -----
plt.xlabel("Confidence")
plt.ylabel("F1 score")
plt.title("F1-Confidence Curve")
plt.grid(False)
plt.ylim(0, 1)
plt.xlim(0, 1)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()

save_path = os.path.join(TARGET_FOLDER, "VALID_Mask_F1_curve.png")
os.makedirs(TARGET_FOLDER, exist_ok=True)
plt.savefig(save_path, dpi=600, bbox_inches="tight")

# Validation Metrics

In [ ]:
# Pixel-level inference
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve, auc
import numpy as np
import torch
import pandas as pd


def best_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return thresholds[np.argmax(f1[:-1])]

def compute_metrics_at_best_f1(all_true, all_probs):
    results = {}
    labels = {0: "IM", 1: "OM"}
    classes = [0, 1]

    for ch in [0, 1]:
        y_true = all_true[ch]
        y_prob = all_probs[ch]
        thresh = best_threshold(y_true, y_prob)
        y_pred = (y_prob >= thresh).astype(np.uint8)

        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))

        precision = tp / (tp + fp + 1e-8)
        recall    = tp / (tp + fn + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        ap50      = average_precision_score(y_true, y_prob)

        results[labels[ch]] = {
            "Mask Precision": round(precision, 3),
            "Mask Recall":    round(recall, 3),
            "mAP50":          round(ap50, 3),
            "F1 Score":       round(f1, 3),
            "Best Threshold": round(float(thresh), 3),
        }

    # Overall
    y_true_all = np.concatenate([all_true[0], all_true[1]])
    y_prob_all  = np.concatenate([all_probs[0], all_probs[1]])
    thresh = best_threshold(y_true_all, y_prob_all)
    y_pred_all = (y_prob_all >= thresh).astype(np.uint8)

    tp = np.sum((y_pred_all == 1) & (y_true_all == 1))
    fp = np.sum((y_pred_all == 1) & (y_true_all == 0))
    fn = np.sum((y_pred_all == 0) & (y_true_all == 1))

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    ap50      = average_precision_score(y_true_all, y_prob_all)

    results["all"] = {
        "Mask Precision": round(precision, 3),
        "Mask Recall":    round(recall, 3),
        "mAP50":          round(ap50, 3),
        "F1 Score":       round(f1, 3),
        "Best Threshold": round(float(thresh), 3),
    }

    return pd.DataFrame(results).T

metrics_df = compute_metrics_at_best_f1(all_true, all_probs)
print("\n── Detectron2 Validation Metrics @Best F1 ──")
print(metrics_df.to_string())